# Ungrouped SEACells metacells - how pure are they?

`build_metacells` in `src/model/data_utils.py` deliberately groups cells by
`(cell_type, cytokine)` *before* clustering, which makes every metacell pure by
construction. This notebook does the opposite: it runs **one global SEACells fit on all
cells at once**, with no grouping whatsoever, and then measures how pure the resulting
metacells actually are:

* does a metacell contain cells of only **one cell type**?
* does it contain cells of only **one condition** (cytokine)?
* does it contain only one **(cell_type, cytokine) combination**, i.e. both at once?

Purity numbers are only meaningful against a null, so every metric is also computed for a
label-permutation baseline (same metacell sizes, labels shuffled across cells), which is
the purity we would get from clustering that carries no information about the labels.

## 0. Imports

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt

from SEACells.core import SEACells, summarize_by_SEACell

from src.model.data_utils import load_data

sc.settings.verbosity = 1

## 1. Configuration

**Scale warning.** The grouped version of `build_metacells` gets away with the dense CPU
SEACells solver because each `(cell_type, cytokine)` group is small. A global fit is a
different beast: the solver holds a dense `n_SEACells x n_cells` assignment matrix, so
running on the full donor (a few hundred thousand cells, hence >10k archetypes) is not
feasible in memory. `SUBSAMPLE_N` therefore subsamples cells uniformly at random -
uniformly, *not* stratified, so the global composition (and hence the purity question) is
preserved. Set it to `None` to attempt the full data set, and expect the runtime and
memory to grow quadratically-ish. `USE_SPARSE=True` (CPU-only) trades speed for memory if
you want to push `SUBSAMPLE_N` higher.

In [ ]:
DATASET_NAME = "parse"
TOP_GENES = 2000
DONOR = ["Donor1"]

CELLS_PER_METACELL = 20   # same target as build_metacells -> n_SEACells = n_cells // 20
N_NEIGHBORS = 15          # same kNN graph size as build_metacells
MAX_ITER = 100            # SEACells Franke-Wolfe iterations
USE_REP = "X_pca"

SUBSAMPLE_N = 20_000      # None = all cells (see warning above)
SEED = 0

USE_GPU = True           # SEACells.gpu backend (needs a CUDA torch)
USE_SPARSE = False        # CPU-only sparse solver: less memory, slower

CELL_TYPE_KEY = "cell_type"   # the condition key comes from load_data (labels_key)
N_PERMUTATIONS = 5            # label-shuffling baseline repeats

result_dir = "/home/jhoefer/sandbox/results/metacell_purity/"
plot_dir = os.path.join(result_dir, "plots")
csv_dir = os.path.join(result_dir, "csv")
os.makedirs(plot_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)

rng = np.random.default_rng(SEED)

## 2. Load the data (no grouping, no splitting)

`log_transform=False` because - exactly as in `build_metacells` - the embedding SEACells
clusters on is always recomputed from `layers["counts"]` below, so log-transforming the
full matrix here would be wasted work.

In [ ]:
adata, labels_key, control_label = load_data(
    DATASET_NAME,
    TOP_GENES,
    log_transform=False,
    cell_types=None,   # every cell type
    donors=DONOR,
)

print(adata)
print(f"labels_key={labels_key!r}, control_label={control_label!r}")

In [ ]:
# Composition of the data we are about to cluster: this is what sets the null purity.
composition = pd.crosstab(adata.obs[CELL_TYPE_KEY], adata.obs[labels_key])
print(f"{adata.n_obs} cells, "
      f"{adata.obs[CELL_TYPE_KEY].nunique()} cell types x "
      f"{adata.obs[labels_key].nunique()} conditions = "
      f"{(composition > 0).to_numpy().sum()} non-empty combinations")
composition

In [ ]:
if SUBSAMPLE_N is not None and SUBSAMPLE_N < adata.n_obs:
    keep = rng.choice(adata.n_obs, size=SUBSAMPLE_N, replace=False)
    keep.sort()
    adata = adata[keep].copy()
    print(f"Subsampled to {adata.n_obs} cells (uniformly at random, seed={SEED}).")
else:
    print(f"Using all {adata.n_obs} cells.")

# The combination is the thing we ultimately care about: a metacell is usable for the
# models downstream only if it is pure in cell_type AND condition simultaneously.
LABEL_KEYS = [CELL_TYPE_KEY, labels_key]  # the two things a metacell should not mix
COMBO_KEY = "combo"
adata.obs[COMBO_KEY] = (
    adata.obs[CELL_TYPE_KEY].astype(str) + " | " + adata.obs[labels_key].astype(str)
).astype("category")

ALL_KEYS = LABEL_KEYS + [COMBO_KEY]
adata.obs[ALL_KEYS].nunique()

## 3. Embedding

Identical recipe to `build_metacells`: log-normalized counts -> 50 PCs, stored in
`obsm['X_pca']`. The only difference is that this is computed once over *all* cells rather
than once per group.

In [ ]:
counts_adata = ad.AnnData(X=adata.layers["counts"].copy())
sc.pp.normalize_total(counts_adata, target_sum=1e4)
sc.pp.log1p(counts_adata)
sc.pp.pca(counts_adata, n_comps=50)
adata.obsm[USE_REP] = counts_adata.obsm["X_pca"]
del counts_adata

adata.obsm[USE_REP].shape

## 4. One global SEACells fit

In [ ]:
n_metacells = adata.n_obs // CELLS_PER_METACELL
print(f"Fitting {n_metacells} SEACells on {adata.n_obs} cells "
      f"(~{CELLS_PER_METACELL} cells per metacell), no grouping.")

model = SEACells(
    adata,
    build_kernel_on=USE_REP,
    n_SEACells=n_metacells,
    n_neighbors=N_NEIGHBORS,
    use_gpu=USE_GPU,
    use_sparse=USE_SPARSE,
    verbose=False,
)

t0 = time.time()
model.construct_kernel_matrix()
model.initialize_archetypes()

try:
    model.fit(min_iter=10, max_iter=MAX_ITER)
    converged = True
except RuntimeWarning as err:
    # SEACells raises (rather than warns) when it hits max_iter; A_ is still usable,
    # the assignments are just the last iterate.
    converged = False
    print(f"NOT CONVERGED after {MAX_ITER} iterations: {err}")

print(f"SEACells fit took {(time.time() - t0) / 60:.1f} min (converged={converged}).")

In [ ]:
adata.obs["SEACell"] = model.get_hard_assignments()["SEACell"]

sizes = adata.obs["SEACell"].value_counts()
print(f"{sizes.size} non-empty metacells out of {n_metacells} requested")
print(f"cells per metacell: min={sizes.min()}, median={sizes.median():.0f}, "
      f"mean={sizes.mean():.1f}, max={sizes.max()}")

adata.obs[["SEACell"] + ALL_KEYS].to_csv(os.path.join(csv_dir, "cell_assignments.csv"))

## 5. Purity

Per metacell and per label key:

* `purity` - fraction of its cells carrying the most abundant label (1.0 = pure),
* `n_labels` - how many distinct labels it mixes,
* `entropy` - Shannon entropy of its label composition, normalized by `log(n_labels_total)`
  so 0 = pure and 1 = as mixed as the label vocabulary allows.

In [ ]:
def purity_table(obs, keys, seacell_key="SEACell"):
    """Per-metacell purity / mixing statistics for each label key."""
    out = {"n_cells": obs.groupby(seacell_key, observed=True).size()}

    for key in keys:
        counts = pd.crosstab(obs[seacell_key], obs[key])
        fracs = counts.div(counts.sum(axis=1), axis=0)

        n_total_labels = (counts.sum(axis=0) > 0).sum()
        with np.errstate(divide="ignore", invalid="ignore"):
            logp = np.where(fracs > 0, np.log(fracs), 0.0)
        entropy = -(fracs.to_numpy() * logp).sum(axis=1) / np.log(n_total_labels)

        out[f"{key}_dominant"] = counts.idxmax(axis=1)
        out[f"{key}_purity"] = fracs.max(axis=1)
        out[f"{key}_n_labels"] = (counts > 0).sum(axis=1)
        out[f"{key}_entropy"] = pd.Series(entropy, index=counts.index)

    table = pd.DataFrame(out)
    table.index.name = seacell_key
    return table


purity = purity_table(adata.obs, ALL_KEYS)
purity.head(10)

In [ ]:
def summarize(purity, keys, weight_col="n_cells"):
    """Metacell-level and cell-level purity summary, one row per label key."""
    rows = []
    w = purity[weight_col]
    for key in keys:
        p = purity[f"{key}_purity"]
        rows.append({
            "label": key,
            "frac_metacells_pure": float((p == 1.0).mean()),
            "frac_metacells_purity_ge_0.9": float((p >= 0.9).mean()),
            "frac_metacells_purity_ge_0.75": float((p >= 0.75).mean()),
            "mean_purity": float(p.mean()),
            "median_purity": float(p.median()),
            # cell-weighted: fraction of *cells* sitting in a metacell whose dominant
            # label is their own - i.e. how much of the data a "majority vote" preserves
            "cell_weighted_purity": float((p * w).sum() / w.sum()),
            "frac_cells_in_pure_metacells": float(w[p == 1.0].sum() / w.sum()),
            "mean_n_labels": float(purity[f"{key}_n_labels"].mean()),
            "max_n_labels": int(purity[f"{key}_n_labels"].max()),
            "mean_entropy": float(purity[f"{key}_entropy"].mean()),
        })
    return pd.DataFrame(rows).set_index("label")


summary = summarize(purity, ALL_KEYS)
summary.round(3)

### 5.1 Baseline: what purity would pure chance give?

If one cell type dominates the data, even a label-blind clustering looks fairly "pure".
Shuffling the labels across cells while keeping the metacell assignment fixed gives the
purity attributable to composition alone; the gap between the two is the part SEACells
actually earned.

In [ ]:
baselines = []
for _ in range(N_PERMUTATIONS):
    shuffled = adata.obs[["SEACell"]].copy()
    perm = rng.permutation(adata.n_obs)
    for key in ALL_KEYS:
        shuffled[key] = adata.obs[key].to_numpy()[perm]
    baselines.append(summarize(purity_table(shuffled, ALL_KEYS), ALL_KEYS))

baseline = sum(baselines) / len(baselines)

comparison = pd.concat(
    {"observed": summary, "shuffled_labels": baseline, "gain": summary - baseline},
    axis=1,
)
comparison.round(3)

In [ ]:
summary.to_csv(os.path.join(csv_dir, "purity_summary.csv"))
baseline.to_csv(os.path.join(csv_dir, "purity_summary_shuffled.csv"))
purity.to_csv(os.path.join(csv_dir, "purity_per_metacell.csv"))

## 6. Plots

In [ ]:
fig, axes = plt.subplots(1, len(ALL_KEYS), figsize=(5 * len(ALL_KEYS), 4), sharey=True)

bins = np.linspace(0, 1, 41)
for axis, key in zip(np.atleast_1d(axes), ALL_KEYS):
    axis.hist(purity[f"{key}_purity"], bins=bins, color="#4C72B0",
              label="observed", alpha=0.85)
    axis.axvline(baseline.loc[key, "mean_purity"], color="#C44E52", ls="--",
                 label=f"shuffled mean ({baseline.loc[key, 'mean_purity']:.2f})")
    axis.axvline(summary.loc[key, "mean_purity"], color="#55A868", ls="-",
                 label=f"observed mean ({summary.loc[key, 'mean_purity']:.2f})")
    axis.set_title(f"{key}\n{summary.loc[key, 'frac_metacells_pure']:.1%} of metacells 100% pure")
    axis.set_xlabel("dominant-label fraction")
    axis.legend(fontsize=8)
np.atleast_1d(axes)[0].set_ylabel("# metacells")

fig.suptitle("Purity of ungrouped SEACells metacells", y=1.03)
fig.tight_layout()
fig.savefig(os.path.join(plot_dir, "purity_histograms.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(purity["n_cells"], purity[f"{COMBO_KEY}_purity"], s=8, alpha=0.4,
                color="#4C72B0")
axes[0].set_xlabel("cells in metacell")
axes[0].set_ylabel("(cell_type, condition) purity")
axes[0].set_title("Purity vs metacell size")

width = 0.8 / len(ALL_KEYS)
max_labels = int(purity[[f"{k}_n_labels" for k in ALL_KEYS]].to_numpy().max())
positions = np.arange(1, max_labels + 1)
for i, key in enumerate(ALL_KEYS):
    counts = purity[f"{key}_n_labels"].value_counts().reindex(positions, fill_value=0)
    axes[1].bar(positions + i * width - 0.4, counts / len(purity), width=width, label=key)
axes[1].set_xlabel("# distinct labels mixed in one metacell")
axes[1].set_ylabel("fraction of metacells")
axes[1].set_title("How many labels does a metacell mix?")
axes[1].legend()

fig.tight_layout()
fig.savefig(os.path.join(plot_dir, "purity_vs_size.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Composition of the 30 most mixed metacells - shows *what* gets mixed with what.
worst = purity.sort_values(f"{COMBO_KEY}_purity").head(30).index

fig, axes = plt.subplots(2, 1, figsize=(12, 9))
for axis, key in zip(axes, LABEL_KEYS):
    counts = pd.crosstab(adata.obs["SEACell"], adata.obs[key]).loc[worst]
    fracs = counts.div(counts.sum(axis=1), axis=0)
    fracs = fracs.loc[:, fracs.sum(axis=0) > 0]

    bottom = np.zeros(len(fracs))
    colors = plt.cm.tab20(np.linspace(0, 1, fracs.shape[1]))
    for color, label in zip(colors, fracs.columns):
        axis.bar(range(len(fracs)), fracs[label], bottom=bottom, color=color, label=label)
        bottom += fracs[label].to_numpy()

    axis.set_xticks(range(len(fracs)))
    axis.set_xticklabels(fracs.index, rotation=90, fontsize=7)
    axis.set_ylabel(f"{key} fraction")
    axis.set_title(f"{key} composition of the 30 least pure metacells")
    axis.legend(fontsize=7, ncol=2, bbox_to_anchor=(1.01, 1), loc="upper left")

fig.tight_layout()
fig.savefig(os.path.join(plot_dir, "worst_metacell_composition.png"), dpi=150,
            bbox_inches="tight")
plt.show()

### 6.1 Cell type vs condition, separately

The two questions can fail very differently: transcriptomic clustering tracks cell
identity strongly, whereas a cytokine perturbation may barely move a cell in PCA space.
This breaks the condition purity down within each dominant cell type.

In [ ]:
by_cell_type = (
    purity.groupby(f"{CELL_TYPE_KEY}_dominant", observed=True)
    .agg(
        n_metacells=("n_cells", "size"),
        n_cells=("n_cells", "sum"),
        cell_type_purity=(f"{CELL_TYPE_KEY}_purity", "mean"),
        condition_purity=(f"{labels_key}_purity", "mean"),
        frac_condition_pure=(f"{labels_key}_purity", lambda p: float((p == 1.0).mean())),
        combo_purity=(f"{COMBO_KEY}_purity", "mean"),
    )
    .sort_values("n_metacells", ascending=False)
)
by_cell_type.to_csv(os.path.join(csv_dir, "purity_by_cell_type.csv"))
by_cell_type.round(3)

In [ ]:
fig, axis = plt.subplots(figsize=(max(6, 0.5 * len(by_cell_type)), 4))
x = np.arange(len(by_cell_type))
axis.bar(x - 0.2, by_cell_type["cell_type_purity"], width=0.4, label="cell type purity")
axis.bar(x + 0.2, by_cell_type["condition_purity"], width=0.4, label="condition purity")
axis.set_xticks(x)
axis.set_xticklabels(by_cell_type.index, rotation=90)
axis.set_ylabel("mean purity")
axis.set_title("Purity per dominant cell type")
axis.legend()
fig.tight_layout()
fig.savefig(os.path.join(plot_dir, "purity_by_cell_type.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. The metacell matrix itself

For completeness, the same aggregation `build_metacells` performs (summing raw counts per
metacell), so the ungrouped metacells can be compared against the grouped cache in
`/g/stegle/jhoefer/data/metacells/`. Each metacell is annotated with its *dominant* labels
plus its purity, since without grouping a metacell no longer has a single true label.

In [ ]:
meta_adata = summarize_by_SEACell(adata, summarize_layer="counts")
meta_adata.var = adata.var.copy()  # summarize_by_SEACell only keeps var_names

for column in purity.columns:
    values = purity[column].reindex(meta_adata.obs_names)
    if column.endswith("_dominant"):  # keep h5ad-writable dtypes
        values = values.astype(str).astype("category")
    meta_adata.obs[column] = values.to_numpy()

meta_adata = meta_adata[meta_adata.obs["n_cells"] >= 2].copy()
meta_adata.layers["counts"] = meta_adata.X.copy()

out_path = os.path.join(result_dir, "ungrouped_metacells.h5ad")
meta_adata.write_h5ad(out_path)
print(meta_adata)
print(f"written to {out_path}")

## 8. Verdict

In [ ]:
n_meta = len(purity)
ct, ct_null = summary.loc[CELL_TYPE_KEY], baseline.loc[CELL_TYPE_KEY]
cond, cond_null = summary.loc[labels_key], baseline.loc[labels_key]
combo, combo_null = summary.loc[COMBO_KEY], baseline.loc[COMBO_KEY]

print(f"Ungrouped SEACells on {adata.n_obs} cells -> {n_meta} metacells "
      f"(median {purity['n_cells'].median():.0f} cells each)\n")

print("Are metacells built from a single CELL TYPE?")
print(f"  {ct['frac_metacells_pure']:.1%} of metacells are 100% pure "
      f"(chance: {ct_null['frac_metacells_pure']:.1%})")
print(f"  mean purity {ct['mean_purity']:.3f} (chance: {ct_null['mean_purity']:.3f}), "
      f"mixing {ct['mean_n_labels']:.1f} cell types on average\n")

print("Are metacells built from a single CONDITION?")
print(f"  {cond['frac_metacells_pure']:.1%} of metacells are 100% pure "
      f"(chance: {cond_null['frac_metacells_pure']:.1%})")
print(f"  mean purity {cond['mean_purity']:.3f} (chance: {cond_null['mean_purity']:.3f}), "
      f"mixing {cond['mean_n_labels']:.1f} conditions on average\n")

print("Are metacells built from a single (CELL TYPE, CONDITION) combination?")
print(f"  {combo['frac_metacells_pure']:.1%} of metacells are 100% pure "
      f"(chance: {combo_null['frac_metacells_pure']:.1%})")
print(f"  {combo['frac_cells_in_pure_metacells']:.1%} of all cells end up in such a metacell")
print(f"  mean purity {combo['mean_purity']:.3f} (chance: {combo_null['mean_purity']:.3f}), "
      f"mixing {combo['mean_n_labels']:.1f} combinations on average\n")

print("=> Grouping before clustering (as build_metacells does) is what makes purity 100% "
      "by construction; the numbers above are what you give up by dropping it.")